In [ ]:
from dep_tools.searchers import PystacSearcher
from dep_tools.loaders import OdcLoader
from src.utils import SDBProcessor, S2_BANDS, load_model_bundle
from pathlib import Path
from distributed import Client

from dep_tools.grids import PACIFIC_GRID_10
import joblib

In [ ]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

In [ ]:
catalog = "https://earth-search.aws.element84.com/v1"
collection = "sentinel-2-l2a"

# tile_id = (130, 12)  # Tiny island
tile_id = (64, 20)  # Suva

geobox = PACIFIC_GRID_10.tile_geobox(tile_id)
datetime = "2024"

# model = joblib.load("models/2025_03_12_randomforest_land_mask_30m.joblib")
model, scaler = load_model_bundle(Path("models/2025_04_16d_nn.zip"))

searcher = PystacSearcher(
    catalog=catalog,
    collections=[collection],
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": 50}},
)

loader = OdcLoader(
    bands=S2_BANDS,
    chunks={"x": 3201, "y": 3201},
    groupby="solar_day",
    fail_on_error=False,
)

processor = SDBProcessor(
    model=model,
    scaler=scaler,
    model_tides=True,
    parallelism=4
)

In [ ]:
from dep_tools.namers import S3ItemPath

itempath = S3ItemPath(
    bucket="example-bucket",
    sensor="s2",
    dataset_id="sdb",
    version="9.9.9",
    time="2024-07/2024-12",
)

In [ ]:
items = searcher.search(geobox)

print(f"Found {len(items)} items")

In [ ]:
data = loader.load(items, geobox)

data

In [ ]:
with Client(n_workers=2, threads_per_worker=16, memory_limit="40GB") as client:
    results = processor.process(data)

results

In [ ]:
results.pc_deep.plot.imshow(size=10)

In [ ]:
(results.pc_pred < 0.3).plot.imshow(size=10)

In [ ]:
import folium
from ipyleaflet import basemaps

masked = results.where(results.pc_deep < 0.7)
masked_two = results.where(results.pc_pred > 0.3)

m = folium.Map(tiles=basemaps.Esri.WorldImagery)
m.fit_bounds(results.odc.map_bounds())

for var in results.data_vars:
    cmap = "Blues" if var in ("mean", "median") else "viridis"
    args = {
        "cmap": cmap,
    }
    if "pc" in var:
        args["vmin"] = 0
        args["vmax"] = 1
    results[var].odc.add_to(m, name=var, **args)

masked["mean"].odc.add_to(m, name="mean masked", cmap="Blues")
masked["median"].odc.add_to(m, name="median masked", cmap="Blues")

masked_two["mean"].odc.add_to(m, name="mean masked two", cmap="Blues")
masked_two["median"].odc.add_to(m, name="median masked two", cmap="Blues")

# Add a layer control to the map
folium.LayerControl().add_to(m)

m

In [ ]:
masked_two["median"].odc.write_cog("median_test.tif")